# SpaPath Crohn's disease workflow

This notebook follows one linear workflow: load reference and disease data, validate image embeddings, preprocess and build graphs, learn initial embeddings, cluster, integrate, detect pathological regions, and construct the all-gene disease dataset.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import scanpy as sc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import spapath_model
import spapath_utils

warnings.filterwarnings("ignore")

In [ ]:
DATASET_ID = "CD"
REFERENCE_SECTION = "V11Y24-011_B"
DISEASE_SECTION = "V10A14_143_D"
DEVICE = "cuda"
SEED = 123

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_ROOT / DATASET_ID
OUTPUT_DIR = PROJECT_ROOT / "outputs" / DATASET_ID
FIGURE_DIR = OUTPUT_DIR / "fig"
HE_IMAGE_PATH = (
    PROJECT_ROOT
    / "data"
    / DATASET_ID
    / "V10A14_143_D.tissue_hires_image.png"
)

for output_path in (OUTPUT_DIR, FIGURE_DIR):
    spapath_utils.create_dir(output_path)

REGION_PALETTE = {
    "Pathological regions": "#B6473F",
    "Healthy-like regions": "#6DBBD1",
}

adata_type_map = {
    REFERENCE_SECTION: "ST_with_HE",
    DISEASE_SECTION: "ST_with_HE",
}
sections = list(adata_type_map)

## 1. Read reference and disease data

In [ ]:
reference_adata = sc.read_h5ad(
    PROCESSED_DIR / f"{REFERENCE_SECTION}_adata.h5ad"
)
disease_adata = sc.read_h5ad(
    PROCESSED_DIR / f"{DISEASE_SECTION}_adata.h5ad"
)

datasets = {
    REFERENCE_SECTION: reference_adata,
    DISEASE_SECTION: disease_adata,
}

## 2. Validate image embeddings

In [ ]:
for section, adata in datasets.items():
    if "image_embedding" not in adata.obsm:
        raise KeyError(f"{section} is missing .obsm['image_embedding'].")

## 3. Preprocess data and build graphs

In [ ]:
batch_list = [reference_adata, disease_adata]
adata_full, disease_adata_all_genes = spapath_utils.preprocess(
    adata_list=batch_list,
    adata_type_map=adata_type_map,
    full_num_hvgs=3000,
    min_genes_qc=10,
    min_cells_qc=10,
)

adata_full = spapath_utils.build_graph_GAT_plus(
    adata_full=adata_full,
    adata_type_map=adata_type_map,
    K=8,
    img_threshold=0.0,
)

## 4. Learn initial embeddings

In [ ]:
model = spapath_model.Model(
    adata_full=adata_full,
    adata_type_map=adata_type_map,
    lr_pre=1e-4,
    lr=1e-4,
    n_pre_training_steps=500,
    n_training_steps=300,
    device=DEVICE,
    seed=SEED,
)

adata_full = model.initial_embedding()

## 5. Cluster observations

In [ ]:
adata_full = model.clustering(
    init_res=1.5,
    intopk=40,
)

## 6. Integrate reference and disease data

In [ ]:
adata_full = model.integrate(topk=40)

## 7. Detect pathological regions

In [ ]:
adata_full = spapath_utils.detection(
    adata=adata_full,
    embed="cell_embed",
    section_ids=sections,
    label_core="Pathological regions",
    label_other="Healthy-like regions",
    core_types=None,
    celltype_key=None,
    batch_key="batch",
    seed=SEED,
    neighbors=30,
    threshold=0.05,
    strategy="individual",
)

In [ ]:
spapath_utils.plot_detection_umap(
    adata=adata_full,
    embed="cell_embed",
    section_id=DISEASE_SECTION,
    batch_key="batch",
    label_key="pred_label",
    label_palette=REGION_PALETTE,
    seed=SEED,
    point_size=18,
    save=FIGURE_DIR / f"{DISEASE_SECTION}_detection_umap.png",
)

he_prediction_figure = spapath_utils.plot_prediction_on_he(
    adata=adata_full,
    image_path=HE_IMAGE_PATH,
    section_id=DISEASE_SECTION,
    batch_key="batch",
    label_key="pred_label",
    spatial_key="spatial",
    label_palette=REGION_PALETTE,
    coordinate_scale=1.0,
    point_scale=0.85,
    figsize=(3, 3),
    save=FIGURE_DIR / f"{DISEASE_SECTION}_prediction_on_he.png",
)

## 8. Build the all-gene disease dataset

In [ ]:
disease_data = spapath_utils.build_disease_data(
    adata_full=adata_full,
    disease_adata_all_genes=disease_adata_all_genes,
    disease_section=DISEASE_SECTION,
)